# Transformer hyperparameter grid search

Random search over Transformer hyperparameters on a multi-region dataset of Switzerland, Norway and Iceland (CH, NOR, ISL), run on 2026-05-20.

**This is not the search behind the paper.** The hyperparameters used in the paper (`PARAMS_TRANSFORMER` in `scripts/config_models.py`, hyperparameter table) are the best configuration of the later search on CH, NOR, ISL and FR in `3.1 Pooled_model_grid_search` (log `FD_MODEL/GS_results/pooled_tf_gs_2026-06-12.csv`). The best configuration found here differs from it.

**Steps**
1. Load the stake datasets and their monthly inputs (`build_monthly_cache`, from `1.4 Create_grids`).
2. Read (or, if missing, compute) the pool/holdout glacier split of each region (`glacier_splits.json`) and build one pooled training dataset from the pool glaciers of the three regions, split randomly into 80% training and 20% validation sequences. The holdout glaciers are kept out.
3. Sample 100 valid configurations from the parameter grid and (with `RUN = True`) train one model per configuration, logging its validation loss and RMSE.
4. Read the log and select the configuration with the lowest validation loss.

The pooled datasets are cached in `cfg.dataPath/cache/TF_Europe/TF_REGION/LSTM_cache_TL/Transformer_exp/`, and the grid-search log in `TF_REGION/logs/Transformer_GS/`. The functions used here live in `scripts/dataset/pooled_dataset.py` and `scripts/models/transformer_grid_search.py`.

## Setting up:

In [ ]:
import os

# Silence KeOps (used by geomloss for Sinkhorn distances) before it is imported
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

import torch

# Initialise CUDA before the other imports: changing this order can change the
# GPU results slightly (~1e-7)
if torch.cuda.is_available():
    torch.cuda.init()

import sys
import logging
import warnings
from pathlib import Path

import pandas as pd
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))  # Add root of repo to import MBM
# Shared config (paths, MONTHLY_COLS, STATIC_COLS, VOIS_CLIMATE, paths_multi, ...),
# model settings (PARAMS_TRANSFORMER) and helpers
from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.config_models import *
from papers.Transformer_TF.scripts.dataset import *
from papers.Transformer_TF.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Call config file (data paths, seed, etc.)
cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

# Use the GPU if one is visible
device = torch.device("cuda" if torch.cuda.device_count() > 0 else "cpu")
print(f"Device: {device}")

# Cache folders: glacier splits and logs (BASE_DIR), pooled datasets (CACHE_DIR)
BASE_DIR = Path(cfg.dataPath) / path_cache / "TF_REGION"
CACHE_DIR = BASE_DIR / "LSTM_cache_TL/Transformer_exp"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for datasets: {CACHE_DIR}")

## Data:
### Stakes and monthly inputs:
With `run_flag_by_code = False`, the monthly data are loaded from `WGMS/CrossRegional/monthly_cache/` instead of being recomputed.

In [ ]:
# Stake datasets of all regions configured in RGI_REGIONS (one DataFrame per RGI region)
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

# Regions pooled for the grid search
SOURCE_REGIONS = ['CH', 'NOR', 'ISL']

run_flag_by_code = {code: False for code in SOURCE_REGIONS}  # True = recompute

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

# Monthly data of the source regions (one entry per region)
res_xreg_by_source = {region: monthly_cache[region] for region in SOURCE_REGIONS}

### Pool and holdout glaciers:
Glacier split of each region into pool glaciers (used for training) and holdout glaciers (kept out of the grid search).

The split is read from `TF_REGION/glacier_splits.json` (the split used for the run on 2026-05-20). If the file is missing, it is recomputed with `compute_pool_holdout_splits` (Sinkhorn-optimised, about 20% of the glaciers in the smaller set). A recomputed split can contain different glaciers than the original one, because that was made from an earlier version of the data.

In [ ]:
SPLITS_PATH = BASE_DIR / "glacier_splits.json"

# Compute the Sinkhorn-optimised split if the file is missing (it can take a while)
if not SPLITS_PATH.exists():
    compute_pool_holdout_splits(monthly_cache,
                                SOURCE_REGIONS,
                                monthly_cols=MONTHLY_COLS,
                                static_cols=STATIC_COLS,
                                save_path=SPLITS_PATH,
                                seed=cfg.seed)

splits = load_pool_holdout_splits(SPLITS_PATH)

### Pooled dataset:
Built from the pool glaciers of all source regions, or loaded from `CACHE_DIR` (`force_recompute=False`). A holdout dataset per region is cached alongside it.

In [ ]:
pooled_assets, ds_holdout_by_region = build_pooled_assets_from_cache(
    res_xreg_by_source=res_xreg_by_source,
    splits=splits,
    source_regions=SOURCE_REGIONS,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    seed=cfg.seed,
    val_ratio=0.2,
    cache_dir=CACHE_DIR,
    force_recompute=False,
)
ds_pooled = pooled_assets["ds_train"]
train_idx = pooled_assets["train_idx"]
val_idx = pooled_assets["val_idx"]

In [ ]:
# Sanity checks: sizes, NaNs, coverage of the pool glaciers, no holdout glaciers in the pool
report_pooled_assets(pooled_assets, ds_holdout_by_region, res_xreg_by_source,
                     splits, SOURCE_REGIONS, MONTHLY_COLS, STATIC_COLS)

## Grid search:
### Parameter grid:
Combinations where `nhead` does not divide `d_model`, or `dim_feedforward` < `d_model`, are skipped; 100 of the remaining combinations are sampled.

In [ ]:
# Fixed parameters: input dimensions and sequence length
const_params = {
    "Fm": ds_pooled.Xm.shape[-1],  # 8 monthly inputs
    "Fs": ds_pooled.Xs.shape[-1],  # 3 static inputs
    "two_heads": False,
    "loss_spec": None,
    "T_max": 32,
}

param_grid = {
    # --- transformer encoder ---
    "d_model": [32, 64, 128, 256],
    "nhead": [2, 4, 8],  # filtered: must divide d_model
    "num_layers": [1, 2, 3, 4],
    "dim_feedforward": [64, 128, 256],  # filtered: must be >= d_model
    "dropout": [0.0, 0.1, 0.2],

    # --- static MLP ---
    "static_layers": [1, 2],
    "static_hidden": [64, 128],
    "static_dropout": [0.0, 0.1, 0.2],

    # --- head ---
    "head_dropout": [0.0, 0.05, 0.1],

    # --- optimisation ---
    "lr": [5e-4, 1e-3, 2e-3],
    "weight_decay": [1e-5, 1e-4],
}

N_SAMPLES = 100
sampled_params = sample_transformer_param_sets(
    const_params,
    param_grid,
    n_samples=N_SAMPLES,
    seed=cfg.seed,
)
print(f"Running {len(sampled_params)} random configurations")

### Run:
Trains one model per configuration (up to 150 epochs, early stopping on the validation loss). This takes several hours, so it only runs with `RUN = True`; the log is written to `logs/Transformer_GS/transformer_gs_pooled_<date>.csv`.

In [ ]:
RUN = False
if RUN:
    best_overall, log_filename = run_transformer_grid_search(
        cfg,
        device,
        ds_pooled,
        train_idx,
        val_idx,
        sampled_params,
        models_dir=BASE_DIR / "models/Transformer_GS",
        logs_dir=BASE_DIR / "logs/Transformer_GS",
        epochs=150,
    )

### Results:
Best configuration (lowest validation loss) of the logged grid search. It is compared with `PARAMS_TRANSFORMER`, which comes from the later search in `3.1 Pooled_model_grid_search` and therefore differs.

In [ ]:
gs_logs_dir = BASE_DIR / "logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(gs_logs_dir / "transformer_gs_pooled_2026-05-20.csv")

best_params_gs = best_params_from_gs_log(logs_gs_transformer)
print("\nSame as PARAMS_TRANSFORMER:", best_params_gs == PARAMS_TRANSFORMER)

In [ ]:
# How stable are the hyperparameters among the 10 best configurations?
top_k = summarize_gs_top_k(logs_gs_transformer, k=10)